In [0]:
# imports
import requests
import json
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

In [0]:
dbutils.widgets.text("API_KEY",dbutils.secrets.get(scope='Connectors', key='api-key'))
dbutils.widgets.text("SAS_TOKEN",dbutils.secrets.get(scope='Connectors', key='SAS-token'))
mode = dbutils.widgets.get("mode")

In [0]:
spark.conf.set("fs.azure.account.auth.type.bitcoindatalake.dfs.core.windows.net", "SAS")
spark.conf.set("fs.azure.sas.token.provider.type.bitcoindatalake.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.sas.FixedSASTokenProvider")
spark.conf.set("fs.azure.sas.fixed.token.bitcoindatalake.dfs.core.windows.net", dbutils.widgets.get("SAS_TOKEN"))

In [0]:
# Check if connection is set up
dbutils.fs.ls("abfss://bronze@bitcoindatalake.dfs.core.windows.net/")

#### Function to get data from coingecko API

In [0]:
def get_data_from_api(url, headers, params):
    try:
            response = requests.get(url, headers=headers, params=params)
            if response.status_code == 200:
                return response.json()
            else:
                raise Exception(f"API call failed with status code {response.status_code}")
    except Exception as e:
        print(f"Error: {e}")
        return None

#### Setting the paramaters and headers

In [0]:
end_time = datetime.now()
start_time_day90 = end_time - timedelta(days=90)
start_time_day1 = end_time - timedelta(days=1)
to_timestamp = end_time.timestamp()
url_ohlc = "https://api.coingecko.com/api/v3/coins/bitcoin/ohlc?vs_currency=usd"
url_chart_data = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart/range"
headers = {
    "x-cg-demo-api-key" : dbutils.widgets.get("API_KEY")}
params_ohlc_lastday = {
    "vs_currency": "usd",
    "days": "1"
}
params_chart_lastday = {
    "vs_currency": "usd",
    "from": start_time_day1.timestamp(),
    "to": to_timestamp
}
params_ohlc_last30days  = {
    "vs_currency": "usd",
    "days": "30"
} 
params_chart_last90days = {
    "vs_currency": "usd",
    "from": start_time_day90.timestamp(),
    "to": to_timestamp
}


In [0]:
try:
    mode = dbutils.widgets.get("mode")
    if mode == "incremental_load":
        chart_data_lastday = get_data_from_api(url_chart_data, headers, params_chart_lastday)
        ohlc_data_lastday = get_data_from_api(url_ohlc, headers, params_ohlc_lastday)
        
        ohlc_data_lastday = {"ohlc" : ohlc_data_lastday}
    elif mode == "initial_load":
        chart_data_last90days = get_data_from_api(url_chart_data, headers, params_chart_last90days)
        ohlc_data_last30days = get_data_from_api(url_ohlc, headers, params_ohlc_last30days)
    
        ohlc_data_last30days = {"ohlc" : ohlc_data_last30days}
    else:
        raise Exception("Invalid mode")
except Exception as e:
    print(f"Error: {e}")
    raise   

In [0]:
base_path = "abfss://raw@bitcoindatalake.dfs.core.windows.net"
def save_json(mode, data, file_name):
    if mode == "incremental_load":
        date = datetime.now() - timedelta(days=1)
        date = date.strftime("%Y-%m-%d")
        path = f"{base_path}/{date}/{file_name}.json"
    else:
        path = f"{base_path}/history/{file_name}.json"
    dbutils.fs.put(path, json.dumps(data), True)
    print(f"file saved to {path}")

In [0]:
def load_to_datalake(mode):
    if mode == "incremental_load":
        save_json(mode, ohlc_data_lastday, "raw_ohlc_lastday")
        save_json(mode, chart_data_lastday, "raw_chart_lastday")
    else:
        save_json(mode, ohlc_data_last30days, "raw_ohlc_last30days")
        save_json(mode, chart_data_last90days, "raw_chart_last90days")


In [0]:
load_to_datalake(mode)

### Dim tables

### dim_coin Table

In [0]:
if mode == "initial_load":
    url = f"https://api.coingecko.com/api/v3/coins/bitcoin"
    data = get_data_from_api(url, headers, None)

    # Assuming 'data' is your dictionary/JSON object
    coin_id = data.get("id")
    ticker_symbol = data.get("symbol")


    # Extracting year from genesis_date
    genesis_date = data.get("genesis_date")
    founded_year = genesis_date.split("-")[0] if genesis_date else None

    # Determining if active (checking for recent updates or status)
    is_active = data.get("market_data") is not None

    dim_coin = {
        "coin_key":1,
        "coin_id": coin_id,
        "ticker_symbol": ticker_symbol,
        "founded_year": founded_year,
        "is_active": is_active,
        "source": "coingecko"
    }
    save_json(mode, dim_coin, "raw_coin")

### dim_currency Table

In [0]:
# I am hardcoding at the moment the values of the USD currency as for now I want to focus on 1 currency
# When the time is right to scale up, I am gonna find a better way to integrate other currencies
if mode == "initial_load":
    currency_data = {
        "currency_key": 1,
        "currency_code": "USD",
        "currency_name": "United States Dollar",
        "currency_symbol": "$",
        "is_active": True
    }
    save_json(mode, currency_data, "raw_currency")

dim_date is gonna be created from the fact_tables (chart_data and ohlc) in the silver layer